In [ ]:


import numpy as np
import pandas as pd
import getpass
from pathlib import Path
import os
from datetime import datetime

pd.set_option('display.max_columns', None)


## File paths ---

user = getpass.getuser()
path_users = Path.home()

path_sp = path_users / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents'
path_raw = path_sp / 'Process Revamp' / 'Task 9. Collect new data' / 'NBI'

# path_bridge = os.path.join(path_sp, 'Products', 'RHNA', 'New Data Collected', 'Bridge Conditions')
# path_data = os.path.join(path_bridge, 'California Bridge Data', 'CA24.txt')




In [ ]:
# Get the full bridge dataset that we will need later on 

def bridge_data(url, county_map):
    # Load data by URL, the URLs are all the same and we use the delimited comma version
    data = pd.read_csv(url, sep=",", dtype=str)

    # We only keep the IDs that are in our county map
    df_bridges = data[data['COUNTY_CODE_003'].astype(str).isin(county_map.keys())].copy()

    return df_bridges


year_list = range(2000,2025)

county_map = {
    '067': 'Sacramento',
    '061': 'Placer',
    '115': 'Yuba',
    '113': 'Yolo',
    '017': 'El Dorado',
    '101': 'Sutter'
}

df_list = []

for year in year_list:
    try:
        # Regular URL: https://www.fhwa.dot.gov/bridge/nbi/2005/delimited/CA05.txt
        url = ["https://www.fhwa.dot.gov/bridge/nbi/", "/delimited/"]

        year_new = str(year)
        year_suffix = "CA" + year_new[-2:] + ".txt"

        url.insert(1, year_new)
        url.insert(3, year_suffix)
        url = ''.join(url)
        print(f"Now collecting bridge data for the year: {year}")
        df_list.append(bridge_data(url, county_map))

    except Exception as e:
        print(f"Issue occurred for the year, {year}: {e}")
        break  # Stops the code

basic_bridge = pd.concat(df_list)

In [ ]:
basic_bridge.head()

***

## **Bridge Condition by County**

***

In [ ]:
# Bridge condition function, this is all you need to pull a single year of data

def bridge_maker(url, county_map, year):
    # Load data by URL, the URLs are all the same and we use the delimited comma version
    data = pd.read_csv(url, sep=",", dtype=str)

    # We only keep the IDs that are in our county map
    df_bridges = data[data['COUNTY_CODE_003'].astype(str).isin(county_map.keys())].copy()

    # Years before 2020 do not have bridge condition, or DECK_AREA thus we have to make our own
    if year < 2020:
        cols = ['DECK_COND_058', 'SUPERSTRUCTURE_COND_059', 'SUBSTRUCTURE_COND_060', 'CULVERT_COND_062']
        df_bridges.loc[:, cols] = df_bridges[cols].replace('N', np.nan).apply(pd.to_numeric)
        df_bridges.loc[:, 'LOWEST_RATING'] = df_bridges[cols].min(axis=1, skipna=True)
        df_bridges = df_bridges.dropna(subset=['LOWEST_RATING'])

        # These conditions and choices were what Ian Schwerzenberg at DVRPC used
        conditions = [
            df_bridges['LOWEST_RATING'] >= 7,
            df_bridges['LOWEST_RATING'].between(5, 6, inclusive='both'),
            df_bridges['LOWEST_RATING'] < 5
        ]
        choices = ['G', 'F', 'P']
        df_bridges.loc[:, 'BRIDGE_CONDITION'] = np.select(conditions, choices, default='Unknown')

        # Convert to feet 

        # - If the deck width in feet doesn't equal 0, multiply the structure length in feet by the deck width in feet
        # - Otherwise, if the deck width in feet does equal 0, multiply the structure length in feet by the approach roadway width in feet

        df_bridges['STRUCTURE_LEN_MT_049'] = df_bridges['STRUCTURE_LEN_MT_049'].astype(float) * 3.281
        df_bridges['DECK_WIDTH_MT_052'] = df_bridges['DECK_WIDTH_MT_052'].astype(float) * 3.281
        df_bridges['APPR_WIDTH_MT_032'] = df_bridges['APPR_WIDTH_MT_032'].astype(float) * 3.281

        # Now, actually calculate the deck area.
        df_bridges.loc[:, 'DECK_AREA'] = df_bridges['STRUCTURE_LEN_MT_049'] * df_bridges['DECK_WIDTH_MT_052']
        df_bridges.loc[df_bridges['DECK_WIDTH_MT_052'] == 0, 'DECK_AREA'] = df_bridges['STRUCTURE_LEN_MT_049'] * df_bridges['APPR_WIDTH_MT_032']

    # For the basic Bridge Conditions dataframe, we only need these columns.
    df_bridges = df_bridges[['COUNTY_CODE_003', 'BRIDGE_CONDITION', 'DECK_AREA', ]]
    df_bridges['DECK_AREA'] = pd.to_numeric(df_bridges['DECK_AREA'], errors='coerce')

    # This is just summing the num of bridges per county code, and then we take the mean of the deck area of those
    df_bridges = df_bridges.groupby(['COUNTY_CODE_003', 'BRIDGE_CONDITION']).agg(
        num_bridges=('COUNTY_CODE_003', 'count'),
        deck_area=('DECK_AREA', 'mean')
    ).reset_index()

    # Assigning the year for each one
    df_bridges['year'] = year

    # Mapping the conditions to same format as Delware Valley
    condition_map = {'G': 'Good', 'P': 'Poor', 'F': 'Fair'}
    df_bridges['BRIDGE_CONDITION'] = df_bridges['BRIDGE_CONDITION'].map(condition_map)
    df_bridges['county_name'] = df_bridges['COUNTY_CODE_003'].astype(str).map(county_map)

    df_bridges.rename(columns={'COUNTY_CODE_003': 'county_id', 'BRIDGE_CONDITION': 'condition'}, inplace=True)

    # This is to get the totals for each county
    county_totals = df_bridges.groupby(['county_id', 'county_name']).agg(
        num_bridges=('num_bridges', 'sum'),
        deck_area=('deck_area', 'sum')
    ).reset_index()

    county_totals['condition'] = 'All'
    county_totals['year'] = year

    df_bridges = pd.concat([df_bridges, county_totals], ignore_index=True)
    df_bridges = df_bridges.sort_values(by=['county_id', 'condition'], ascending=[True, True])

    df_bridges = df_bridges[['year', 'county_id', 'county_name', 'condition', 'deck_area', 'num_bridges']]
    # print(f"Bridge condition data for the year: {year}.")

    return df_bridges

In [ ]:
# Create the Bridge Conditions Rating Dataframe

year_list = range(2000,2025)

county_map = {
    '067': 'Sacramento',
    '061': 'Placer',
    '115': 'Yuba',
    '113': 'Yolo',
    '017': 'El Dorado',
    '101': 'Sutter'
}

df_list = []

for year in year_list:
    try:
        # Regular URL: https://www.fhwa.dot.gov/bridge/nbi/2005/delimited/CA05.txt
        url = ["https://www.fhwa.dot.gov/bridge/nbi/", "/delimited/"]

        year_new = str(year)
        year_suffix = "CA" + year_new[-2:] + ".txt"

        url.insert(1, year_new)
        url.insert(3, year_suffix)
        url = ''.join(url)
        print(f"Now collecting bridge data for the year: {year}")
        df_list.append(bridge_maker(url, county_map, year))

    except Exception as e:
        print(f"Issue occurred for the year, {year}: {e}")
        break  # Stops the code

df_bridge_cond = pd.concat(df_list)
display(df_bridge_cond)
### 2019 First year to collect BRIDGE_CONDITION and DECK_AREA

In [ ]:
# Export code

# Export Prep
path_cond = os.path.join(path_bridge, "Bridge Condition by County")

print(''); print('')
print('Converting/exporting Bridge Condition results to SharePoint')

# Export bridge cond
filename = f"Bridge Condition by County - SACOG"
name_out_csv = filename + '.csv'
path_out_csv = os.path.join(path_cond, name_out_csv)
df_bridge_cond.to_csv(path_out_csv, index=False)

# print(f"Excel files exported here:  {path_out_xlsx}");print('')
print(f"Files exported here:  {path_cond}");print('')

***

## **Total and Percent Deficient by County**

***

In [ ]:
# Convert the dataframe to a dictionary mapping 'maintenance_021' to 'owner_type'
# owner_map = pd.read_csv("owner_types.csv")  # Replace with the actual file path
# owner_dict = owner_map.set_index('maintenance_021')['owner_type'].to_dict()

# https://www.fhwa.dot.gov/bridge/mtguide.pdf
owner_map = {
   '01' : 'State Highway Agency'
 , '02' : 'County Highway Agency'
 , '03' : 'Town or Township Highway Agency'
 , '04' : 'City or Municipal Highway Agency'
 , '11': 'State Park, Forest, or Reservation Agency'
 , '12': 'Local Park, Forest, or Reservation Agency'
 , '21': 'Other State Agencies'
 , '25': 'Other Local Agencies'
 , '26': 'Private (other than railroad)'
 , '27': 'Railroad'
 , '31': 'State Toll Authority'
 , '32': 'Local Toll Authority'
 , '60': 'Other Federal Agencies (not listed below)'
 , '61': 'Indian Tribal Government'
 , '62': 'Bureau of Indian Affairs'
 , '63': 'Bureau of Fish and Wildlife'
 , '64': 'U.S. Forest Service'
 , '66': 'National Park Service'
 , '67': 'Tennessee Valley Authority'
 , '68': 'Bureau of Land Management'
 , '69': 'Bureau of Reclamation'
 , '70': 'Corps of Engineers (Civil)'
 , '71': 'Corps of Engineers (Military)'
 , '72': 'Air Force'
 , '73': 'Navy/Marines'
 , '74': 'Army'
 , '75': 'NASA'
 , '76': 'Metropolitan Washington Airports Service'
 , '80': 'Unknown'
}

def bridge_deficient(url, county_map, year, owner_map):
# def bridge_deficient(url, county_map, year):

    # Load data by URL, the URLs are all the same and we use the delimited comma version
    data = pd.read_csv(url, sep=",", dtype=str)

    # We only keep the IDs that are in our county map
    df_bridges = data[data['COUNTY_CODE_003'].astype(str).isin(county_map.keys())].copy()

    if year < 2020:
        # Convert to feet 

        # - If the deck width in feet doesn't equal 0, multiply the structure length in feet by the deck width in feet
        # - Otherwise, if the deck width in feet does equal 0, multiply the structure length in feet by the approach roadway width in feet

        df_bridges['STRUCTURE_LEN_MT_049'] = df_bridges['STRUCTURE_LEN_MT_049'].astype(float) * 3.281
        df_bridges['DECK_WIDTH_MT_052'] = df_bridges['DECK_WIDTH_MT_052'].astype(float) * 3.281
        df_bridges['APPR_WIDTH_MT_032'] = df_bridges['APPR_WIDTH_MT_032'].astype(float) * 3.281

        # Now, actually calculate the deck area.
        df_bridges.loc[:, 'DECK_AREA'] = df_bridges['STRUCTURE_LEN_MT_049'] * df_bridges['DECK_WIDTH_MT_052']
        df_bridges.loc[df_bridges['DECK_WIDTH_MT_052'] == 0, 'DECK_AREA'] = df_bridges['STRUCTURE_LEN_MT_049'] * df_bridges['APPR_WIDTH_MT_032']

    # For the basic Bridge Conditions dataframe, we only need these columns.
    df_bridges = df_bridges[['COUNTY_CODE_003', 'MAINTENANCE_021', 'DECK_AREA']]
    df_bridges['DECK_AREA'] = pd.to_numeric(df_bridges['DECK_AREA'], errors='coerce')

    # This is just summing the num of bridges per county code, and then we take the mean of the deck area of those
    df_bridges = df_bridges.groupby(['COUNTY_CODE_003', 'MAINTENANCE_021']).agg(
        num_bridges=('COUNTY_CODE_003', 'count'),
        deck_area=('DECK_AREA', 'mean')
    ).reset_index()

    # Assigning the year for each one
    df_bridges['year'] = year
    df_bridges['county_name'] = df_bridges['COUNTY_CODE_003'].astype(str).map(county_map)
    df_bridges['MAINTENANCE_021'] = df_bridges['MAINTENANCE_021'].astype(str)
    df_bridges['owner_type'] = df_bridges['MAINTENANCE_021'].map(owner_map)
    # df_bridges['owner_type'] = df_bridges['MAINTENANCE_021'].copy()


    df_bridges.rename(columns={'COUNTY_CODE_003': 'county_id'}, inplace=True)

    # # This is to get the totals for each county
    # county_totals = df_bridges.groupby(['county_id', 'county_name']).agg(
    #     num_bridges=('num_bridges', 'sum'),
    #     deck_area=('deck_area', 'sum')
    # ).reset_index()

    # county_totals['year'] = year

    # df_bridges = pd.concat([df_bridges, county_totals], ignore_index=True)
    df_bridges = df_bridges.groupby(['county_id', 'county_name', 'year', 'owner_type']).agg(
        num_bridges=('num_bridges', 'sum'),
        deck_area=('deck_area', 'sum')
    ).reset_index()

    totals = df_bridges.groupby(['county_id', 'county_name', 'year']).agg(
        num_bridges=('num_bridges', 'sum'),
        deck_area=('deck_area', 'sum')
    ).reset_index()

    totals['owner_type'] = 'All'
    df_bridges = pd.concat([df_bridges, totals], ignore_index=True)
    df_bridges = df_bridges.sort_values(by=['county_id', 'owner_type'], ascending=[True, True])

    df_bridges = df_bridges[['year', 'county_id', 'county_name', 'owner_type', 'deck_area', 'num_bridges']]
    
    # print(f"Bridge condition data for the year: {year}.")

    return df_bridges

In [ ]:
year_list = range(2000,2025)

county_map = {
    '067': 'Sacramento',
    '061': 'Placer',
    '115': 'Yuba',
    '113': 'Yolo',
    '017': 'El Dorado',
    '101': 'Sutter'
}

df_list = []

for year in year_list:
    try:
        # Regular URL: https://www.fhwa.dot.gov/bridge/nbi/2005/delimited/CA05.txt
        url = ["https://www.fhwa.dot.gov/bridge/nbi/", "/delimited/"]

        year_new = str(year)
        year_suffix = "CA" + year_new[-2:] + ".txt"

        url.insert(1, year_new)
        url.insert(3, year_suffix)
        url = ''.join(url)
        print(f"Now collecting bridge data for the year: {year}")
        df_list.append(bridge_deficient(url, county_map, year, owner_map))
        # df_list.append(bridge_deficient(url, county_map, year))

    except Exception as e:
        print(f"Issue occurred for the year, {year}: {e}")
        break  # Stops the code

full_bridge = pd.concat(df_list)

In [ ]:
# Percents now
new_bridge_code = full_bridge_code.copy()
new_bridge_code['deck_area'] = new_bridge_code['deck_area'] / new_bridge_code['deck_area'].sum()
new_bridge_code['num_bridges'] = new_bridge_code['num_bridges'] / new_bridge_code['num_bridges'].sum()
display(new_bridge_code.head())

In [ ]:
# Percents now
new_bridge = full_bridge.copy()
new_bridge['deck_area'] = new_bridge['deck_area'] / new_bridge['deck_area'].sum()
new_bridge['num_bridges'] = new_bridge['num_bridges'] / new_bridge['num_bridges'].sum()
display(new_bridge.head())

In [ ]:
new_bridge_code.owner_type.unique()

In [ ]:
new_bridge.owner_type.unique()

In [ ]:
# Exports

# Percent Def by County

path_percent = os.path.join(path_bridge, "Bridge Deficiency by County")

print(''); print('')
print('Converting/exporting Bridge Deficiency results to SharePoint')

# Export bridge cond
percent_name = f"Percentage Deficient by County - SACOG"
total_name = f"Total Deficient by County - SACOG"
per_name_out_csv = percent_name + '.csv'
tot_name_out_csv = total_name + '.csv'
per_path_out_csv = os.path.join(path_cond, per_name_out_csv)
tot_path_out_csv = os.path.join(path_cond, tot_name_out_csv)

new_bridge.to_csv(per_path_out_csv, index=False)
full_bridge.to_csv(tot_path_out_csv, index = False)

# print(f"Excel files exported here:  {path_out_xlsx}");print('')
print(f"Files exported here:  {path_cond}");print('')

***

# **Code Graveyard**

***

In [ ]:
# WOULD REALLY HAVE LIKED TO GET THIS VERSION WORKING


# Bridge condition function, this is all you need to pull a single year of data

def bridge_maker(data, county_map, year):
    # Years before 2020 do not have bridge condition, or DECK_AREA thus we have to make our own
    df_bridges = data.copy()
    
    if year < 2020:
        cols = ['DECK_COND_058', 'SUPERSTRUCTURE_COND_059', 'SUBSTRUCTURE_COND_060', 'CULVERT_COND_062']
        df_bridges.loc[:, cols] = df_bridges[cols].replace('N', np.nan).apply(pd.to_numeric)
        df_bridges.loc[:, 'LOWEST_RATING'] = df_bridges[cols].min(axis=1, skipna=True)
        df_bridges = df_bridges.dropna(subset=['LOWEST_RATING'])

        # These conditions and choices were what Ian Schwerzenberg at DVRPC used
        conditions = [
            df_bridges['LOWEST_RATING'] >= 7,
            df_bridges['LOWEST_RATING'].between(5, 6, inclusive='both'),
            df_bridges['LOWEST_RATING'] < 5
        ]
        choices = ['G', 'F', 'P']
        df_bridges.loc[:, 'BRIDGE_CONDITION'] = np.select(conditions, choices, default='Unknown')

        # Convert to feet 

        # - If the deck width in feet doesn't equal 0, multiply the structure length in feet by the deck width in feet
        # - Otherwise, if the deck width in feet does equal 0, multiply the structure length in feet by the approach roadway width in feet

        df_bridges['STRUCTURE_LEN_MT_049'] = df_bridges['STRUCTURE_LEN_MT_049'].astype(float) * 3.281
        df_bridges['DECK_WIDTH_MT_052'] = df_bridges['DECK_WIDTH_MT_052'].astype(float) * 3.281
        df_bridges['APPR_WIDTH_MT_032'] = df_bridges['APPR_WIDTH_MT_032'].astype(float) * 3.281

        # Now, actually calculate the deck area.
        df_bridges.loc[:, 'DECK_AREA'] = df_bridges['STRUCTURE_LEN_MT_049'] * df_bridges['DECK_WIDTH_MT_052']
        df_bridges.loc[df_bridges['DECK_WIDTH_MT_052'] == 0, 'DECK_AREA'] = df_bridges['STRUCTURE_LEN_MT_049'] * df_bridges['APPR_WIDTH_MT_032']

    # For the basic Bridge Conditions dataframe, we only need these columns.
    df_bridges = df_bridges[['COUNTY_CODE_003', 'BRIDGE_CONDITION', 'DECK_AREA', ]]
    df_bridges['DECK_AREA'] = pd.to_numeric(df_bridges['DECK_AREA'], errors='coerce')

    # This is just summing the num of bridges per county code, and then we take the mean of the deck area of those
    df_bridges = df_bridges.groupby(['COUNTY_CODE_003', 'BRIDGE_CONDITION']).agg(
        num_bridges=('COUNTY_CODE_003', 'count'),
        deck_area=('DECK_AREA', 'mean')
    ).reset_index()

    # Assigning the year for each one
    df_bridges['year'] = year

    # Mapping the conditions to same format as Delware Valley
    condition_map = {'G': 'Good', 'P': 'Poor', 'F': 'Fair'}
    df_bridges['BRIDGE_CONDITION'] = df_bridges['BRIDGE_CONDITION'].map(condition_map)
    df_bridges['county_name'] = df_bridges['COUNTY_CODE_003'].astype(str).map(county_map)

    df_bridges.rename(columns={'COUNTY_CODE_003': 'county_id', 'BRIDGE_CONDITION': 'condition'}, inplace=True)

    # This is to get the totals for each county
    county_totals = df_bridges.groupby(['county_id', 'county_name']).agg(
        num_bridges=('num_bridges', 'sum'),
        deck_area=('deck_area', 'sum')
    ).reset_index()

    county_totals['condition'] = 'All'
    county_totals['year'] = year

    df_bridges = pd.concat([df_bridges, county_totals], ignore_index=True)
    df_bridges = df_bridges.sort_values(by=['county_id', 'condition'], ascending=[True, True])

    df_bridges = df_bridges[['year', 'county_id', 'county_name', 'condition', 'deck_area', 'num_bridges']]
    # print(f"Bridge condition data for the year: {year}.")

    return df_bridges
# Create the Bridge Conditions Rating Dataframe

year_list = range(2000,2025)

county_map = {
    '067': 'Sacramento',
    '061': 'Placer',
    '115': 'Yuba',
    '113': 'Yolo',
    '017': 'El Dorado',
    '101': 'Sutter'
}

df_list = []

for year in year_list:
    try:
        print(f"Now collecting bridge data for the year: {year}")
        df_list.append(bridge_maker(basic_bridge, county_map, year))

    except Exception as e:
        print(f"Issue occurred for the year, {year}: {e}")
        break  # Stops the code

df_bridge_cond = pd.concat(df_list)
display(df_bridge_cond)
### 2019 First year to collect BRIDGE_CONDITION and DECK_AREA

In [ ]:
# WOULD REALLY HAVE LIKED TO GET THIS VERSION WORKING


# Convert the dataframe to a dictionary mapping 'maintenance_021' to 'owner_type'
owner_map = pd.read_csv("owner_types.csv")  # Replace with the actual file path
owner_dict = owner_map.set_index('maintenance_021')['owner_type'].to_dict()

def bridge_deficient(data, county_map, year, owner_dict):
    df_bridges = data.copy()
    if year < 2020:
        # Convert to feet 

        # - If the deck width in feet doesn't equal 0, multiply the structure length in feet by the deck width in feet
        # - Otherwise, if the deck width in feet does equal 0, multiply the structure length in feet by the approach roadway width in feet

        df_bridges['STRUCTURE_LEN_MT_049'] = df_bridges['STRUCTURE_LEN_MT_049'].astype(float) * 3.281
        df_bridges['DECK_WIDTH_MT_052'] = df_bridges['DECK_WIDTH_MT_052'].astype(float) * 3.281
        df_bridges['APPR_WIDTH_MT_032'] = df_bridges['APPR_WIDTH_MT_032'].astype(float) * 3.281

        # Now, actually calculate the deck area.
        df_bridges.loc[:, 'DECK_AREA'] = df_bridges['STRUCTURE_LEN_MT_049'] * df_bridges['DECK_WIDTH_MT_052']
        df_bridges.loc[df_bridges['DECK_WIDTH_MT_052'] == 0, 'DECK_AREA'] = df_bridges['STRUCTURE_LEN_MT_049'] * df_bridges['APPR_WIDTH_MT_032']

    # For the basic Bridge Conditions dataframe, we only need these columns.
    df_bridges = df_bridges[['COUNTY_CODE_003', 'MAINTENANCE_021', 'DECK_AREA']]
    df_bridges['DECK_AREA'] = pd.to_numeric(df_bridges['DECK_AREA'], errors='coerce')

    # This is just summing the num of bridges per county code, and then we take the mean of the deck area of those
    df_bridges = df_bridges.groupby(['COUNTY_CODE_003', 'MAINTENANCE_021']).agg(
        num_bridges=('COUNTY_CODE_003', 'count'),
        deck_area=('DECK_AREA', 'mean')
    ).reset_index()

    # Assigning the year for each one
    df_bridges['year'] = year
    df_bridges['county_name'] = df_bridges['COUNTY_CODE_003'].astype(str).map(county_map)
    df_bridges['MAINTENANCE_021'] = df_bridges['MAINTENANCE_021'].astype(int)
    df_bridges['owner_type'] = df_bridges['MAINTENANCE_021'].map(owner_dict)


    df_bridges.rename(columns={'COUNTY_CODE_003': 'county_id'}, inplace=True)

    # df_bridges = pd.concat([df_bridges, county_totals], ignore_index=True)
    df_bridges = df_bridges.groupby(['county_id', 'county_name', 'year', 'owner_type']).agg(
        num_bridges=('num_bridges', 'sum'),
        deck_area=('deck_area', 'sum')
    ).reset_index()

    totals = df_bridges.groupby(['county_id', 'county_name', 'year']).agg(
        num_bridges=('num_bridges', 'sum'),
        deck_area=('deck_area', 'sum')
    ).reset_index()

    totals['owner_type'] = 'All'

    df_bridges = pd.concat([df_bridges, totals], ignore_index=True)
    df_bridges = df_bridges.sort_values(by=['county_id', 'owner_type'], ascending=[True, True])
    df_bridges = df_bridges[['year', 'county_id', 'county_name', 'owner_type', 'deck_area', 'num_bridges']]
    
    # print(f"Bridge condition data for the year: {year}.")

    return df_bridges
year_list = range(2000,2025)

bridge_list = []
for year in year_list:
    try:
        bridge_list.append(bridge_deficient(basic_bridge, county_map, year, owner_dict))

    except Exception as e:
        print(f"Issue occurred for the year, {year}: {e}")
        break  # Stops the code

full_bridge = pd.concat(df_list)

In [ ]:
# # Creating the final look for the bridges condition df
# df_bridges['DECK_AREA'] = pd.to_numeric(df_bridges['DECK_AREA'], errors='coerce')

# df_bridges = df_bridges.groupby(['COUNTY_CODE_003', 'BRIDGE_CONDITION']).agg(
#     num_bridges=('COUNTY_CODE_003', 'count'),  # Count occurrences
#     deck_area=('DECK_AREA', 'mean')  # Average of DECK_AREA
# ).reset_index()

# df_bridges['year'] = '2024'

# condition_map = {'G': 'Good', 'P': 'Poor', 'F': 'Fair'}
# df_bridges['BRIDGE_CONDITION'] = df_bridges['BRIDGE_CONDITION'].map(condition_map)
# df_bridges['county_name'] = df_bridges['COUNTY_CODE_003'].astype(str).map(county_map)

# df_bridges.rename(columns={'COUNTY_CODE_003': 'county_id', 'BRIDGE_CONDITION': 'condition'}, inplace=True)

# county_totals = df_bridges.groupby(['county_id', 'county_name']).agg(
#     num_bridges=('num_bridges', 'sum'),
#     deck_area=('deck_area', 'sum')
# ).reset_index()

# county_totals['condition'] = 'All'
# county_totals['year'] = '2024'

# # Concatenate All rows with the original dataframe
# df_bridges = pd.concat([df_bridges, county_totals], ignore_index=True)
# df_bridges = df_bridges[['year', 'county_id', 'county_name', 'condition', 'deck_area', 'num_bridges']]
# display(df_bridges)

In [ ]:
# county_totals = df_bridges.groupby(['county_id', 'county_name']).agg(
#     num_bridges=('num_bridges', 'sum'),
#     deck_area=('deck_area', 'sum')
# ).reset_index()

# county_totals['condition'] = 'All'
# county_totals['year'] = '2024'

In [ ]:
# df_bridges

In [ ]:
# # Mapping COUNTY_CODE_003 to county names
# county_map = {
#     '067': 'Sacramento',
#     '061': 'Placer',
#     '115': 'Yuba',
#     '113': 'Yolo',
#     '017': 'El Dorado',
#     '101': 'Sutter'
# }

# def bridge_maker(url, county_map, year):
#     data = pd.read_csv(url, sep=",", dtype=str)

#     # Only keep the SACOG ones
#     df_bridges = data[data['COUNTY_CODE_003'].astype(str).isin(county_map.keys())]

#     df_bridges = df_bridges.copy()

#     if year < 2020:
#         # # Bridge Condition and Deck Area does not exist for datasets before 2020, so we have to make our own. 
#         # df_bridges[['DECK_COND_058', 'SUPERSTRUCTURE_COND_059', 'SUBSTRUCTURE_COND_060', 'CULVERT_COND_062']] = df_bridges[['DECK_COND_058', 'SUPERSTRUCTURE_COND_059', 'SUBSTRUCTURE_COND_060', 'CULVERT_COND_062']].replace('N', np.nan).apply(pd.to_numeric)
#         # df_bridges['LOWEST_RATING'] = df_bridges[['DECK_COND_058', 'SUPERSTRUCTURE_COND_059', 'SUBSTRUCTURE_COND_060', 'CULVERT_COND_062']].min(axis=1, skipna=True)

#         cols = ['DECK_COND_058', 'SUPERSTRUCTURE_COND_059', 'SUBSTRUCTURE_COND_060', 'CULVERT_COND_062']
#         df_bridges.loc[:, cols] = df_bridges[cols].replace('N', np.nan).apply(pd.to_numeric)
#         df_bridges.loc[:, 'LOWEST_RATING'] = df_bridges[cols].min(axis=1, skipna=True)

#         conditions = [
#         df_bridges['LOWEST_RATING'] >= 7,
#         df_bridges['LOWEST_RATING'].between(5, 6, inclusive='both'),
#         df_bridges['LOWEST_RATING'] < 5
#         ]

#         # Make labels for bridge condition
#         choices = ['G', 'F', 'P']

#         # Apply the labels
#         df_bridges.loc[:, 'BRIDGE_CONDITION'] = np.select(conditions, choices, default='Unknown')
#         df_bridges['STRUCTURE_LEN_MT_049'] = df_bridges['STRUCTURE_LEN_MT_049'].astype(float) * 3.281
#         df_bridges['DECK_WIDTH_MT_052'] = df_bridges['DECK_WIDTH_MT_052'].astype(float) * 3.281
#         df_bridges['APPR_WIDTH_MT_032'] = df_bridges['APPR_WIDTH_MT_032'].astype(float) * 3.281

#         # We calculate deck area off of cols 49 and 52, but if deck width is zero, we use the approximate
#         df_bridges.loc[:, 'DECK_AREA'] = df_bridges['STRUCTURE_LEN_MT_049'] * df_bridges['DECK_WIDTH_MT_052']
#         df_bridges.loc[df_bridges['DECK_WIDTH_MT_052'] == 0, 'DECK_AREA'] = df_bridges['STRUCTURE_LEN_MT_049'] * df_bridges['APPR_WIDTH_MT_032']

#     df_bridges = df_bridges[['COUNTY_CODE_003', 'BRIDGE_CONDITION', 'DECK_AREA']]

#     # Creating the final look for the bridges condition df
#     df_bridges['DECK_AREA'] = pd.to_numeric(df_bridges['DECK_AREA'], errors='coerce')

#     df_bridges = df_bridges.groupby(['COUNTY_CODE_003', 'BRIDGE_CONDITION']).agg(
#         num_bridges=('COUNTY_CODE_003', 'count'),  # Count occurrences
#         deck_area=('DECK_AREA', 'mean')  # Average of DECK_AREA
#     ).reset_index()

#     df_bridges['year'] = year

#     condition_map = {'G': 'Good', 'P': 'Poor', 'F': 'Fair'}
#     df_bridges['BRIDGE_CONDITION'] = df_bridges['BRIDGE_CONDITION'].map(condition_map)
#     df_bridges['county_name'] = df_bridges['COUNTY_CODE_003'].astype(str).map(county_map)

#     df_bridges.rename(columns={'COUNTY_CODE_003': 'county_id', 'BRIDGE_CONDITION': 'condition'}, inplace=True)

#     county_totals = df_bridges.groupby(['county_id', 'county_name']).agg(
#         num_bridges=('num_bridges', 'sum'),
#         deck_area=('deck_area', 'sum')
#     ).reset_index()

#     county_totals['condition'] = 'All'
#     county_totals['year'] = year

#     # Concatenate All rows with the original dataframe
#     df_bridges = pd.concat([df_bridges, county_totals], ignore_index=True)
#     df_bridges = df_bridges[['year', 'county_id', 'county_name', 'condition', 'deck_area', 'num_bridges']]
#     print(f"Bridge condition data for the year: {year}.")
#     display(df_bridges.head(5))

#     return df_bridges

# tester = bridge_maker(url, county_map, 2000)

In [ ]:
# test = pd.read_csv("https://www.fhwa.dot.gov/bridge/nbi/2011/delimited/CA11.txt", sep=",", dtype=str)

In [ ]:
# test

In [ ]:

# # Map the owner types
# mapping_df = pd.read_csv(path_map)  # Replace with actual path

# maintenance_map = dict(zip(mapping_df["maintenance_021"], mapping_df["owner_type"]))

# maintenance_map

In [ ]:
# path_map = os.path.join(path_bridge, 'owner_types.csv')

# def bridge_owner(url, county_map, year):
#     # Read the CSV file into a DataFrame, treating all columns as strings
#     data = pd.read_csv(url, sep=",", dtype=str)
    
#     # Filter the data to include only counties in the county_map
#     df_bridges = data[data['COUNTY_CODE_003'].astype(str).isin(county_map.keys())].copy()
    
#     # Retain the MAINTENANCE_021 column for later use
#     df_bridges = df_bridges[['COUNTY_CODE_003', 'MAINTENANCE_021', 'DECK_COND_058', 'SUPERSTRUCTURE_COND_059',
#                              'SUBSTRUCTURE_COND_060', 'CULVERT_COND_062', 'STRUCTURE_LEN_MT_049',
#                              'DECK_WIDTH_MT_052', 'APPR_WIDTH_MT_032']]
    
#     # Convert structure dimensions from meters to feet
#     df_bridges['STRUCTURE_LEN_MT_049'] = df_bridges['STRUCTURE_LEN_MT_049'].astype(float) * 3.281
#     df_bridges['DECK_WIDTH_MT_052'] = df_bridges['DECK_WIDTH_MT_052'].astype(float) * 3.281
#     df_bridges['APPR_WIDTH_MT_032'] = df_bridges['APPR_WIDTH_MT_032'].astype(float) * 3.281
    
#     # Calculate deck area using structure length and deck width
#     df_bridges.loc[:, 'DECK_AREA'] = df_bridges['STRUCTURE_LEN_MT_049'] * df_bridges['DECK_WIDTH_MT_052']
    
#     # If deck width is zero, use the approximate width instead
#     df_bridges.loc[df_bridges['DECK_WIDTH_MT_052'] == 0, 'DECK_AREA'] = (
#         df_bridges['STRUCTURE_LEN_MT_049'] * df_bridges['APPR_WIDTH_MT_032']
#     )
    
#     # Keep only relevant columns for aggregation
#     df_bridges = df_bridges[['COUNTY_CODE_003', 'DECK_AREA', 'MAINTENANCE_021']]

#     # Map the owner types
#     mapping_df = pd.read_csv(path_map)  # Replace with actual path

#     maintenance_map = dict(zip(mapping_df["maintenance_021"], mapping_df["owner_type"]))

#     df_bridges["MAINTENANCE_021"] = df_bridges["MAINTENANCE_021"].map(maintenance_map)

#     # Convert deck area to numeric
#     df_bridges['DECK_AREA'] = pd.to_numeric(df_bridges['DECK_AREA'], errors='coerce')
    
#     # Aggregate data by county and bridge condition
#     df_bridges = df_bridges.groupby(['COUNTY_CODE_003', 'MAINTENANCE_021']).agg(
#         num_bridges=('COUNTY_CODE_003', 'count'),  # Count occurrences
#         deck_area=('DECK_AREA', 'mean')  # Average of DECK_AREA
#     ).reset_index()
    
#     # Add year column
#     df_bridges['year'] = year
    
#     # Map county codes to county names
#     df_bridges['county_name'] = df_bridges['COUNTY_CODE_003'].astype(str).map(county_map)
    
#     # Rename columns for clarity
#     df_bridges.rename(columns={'COUNTY_CODE_003': 'county_id'}, inplace=True)
    
#     # # Calculate totals for each county across all conditions
#     # county_totals = df_bridges.groupby(['county_id', 'county_name']).agg(
#     #     num_bridges=('num_bridges', 'sum'),
#     #     deck_area=('deck_area', 'sum')
#     # ).reset_index()
    
#     # # Add 'All' category to represent totals
#     # county_totals['condition'] = 'All'
#     # county_totals['year'] = year
    
#     # # Display county totals
#     # display(county_totals)
    
#     # # Append totals to the main DataFrame
#     # df_bridges = pd.concat([df_bridges, county_totals], ignore_index=True)
    
#     # # Sort data for better readability
#     # df_bridges = df_bridges.sort_values(by=['county_id', 'MAINTENANCE_021'], ascending=[True, True])
    
#     # # Reorder columns for the final output
#     # df_bridges = df_bridges[['year', 'county_id', 'county_name', 'deck_area', 'num_bridges', 'MAINTENANCE_021']]
    
#     # print(f"Bridge condition data for the year: {year}.")
#     # display(df_bridges.head(5))
    
#     return df_bridges

# test = bridge_owner("https://www.fhwa.dot.gov/bridge/nbi/2011/delimited/CA11.txt", county_map, 2011)

In [ ]:
# df_bridge_cond[df_bridge_cond['year'] == 2024]